# Aegis - Phase 1: Evaluation Harness & Baselines

Assembles the corpus, trains the RJD baselines + Aegis-Fast, prints the comparison + robustness tables.

**Setup:** Add Data -> Upload the Aegis repo zip. Internet ON (+ GPU for the open guard). Run All.

In [ ]:
import sys, os, glob, zipfile, shutil
shutil.rmtree('/kaggle/working/_aegis_src', ignore_errors=True)   # clear any stale extracted copy
def find_aegis_root():
    hits = sorted(glob.glob('/kaggle/input/**/aegis/__init__.py', recursive=True))   # prefer attached folder
    if hits: return os.path.dirname(os.path.dirname(os.path.abspath(hits[0])))
    for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):                    # else extract attached zip fresh
        try:
            with zipfile.ZipFile(z) as zf:
                if any(n.endswith('aegis/__init__.py') for n in zf.namelist()): zf.extractall('/kaggle/working/_aegis_src')
        except Exception: pass
    hits = glob.glob('/kaggle/working/_aegis_src/**/aegis/__init__.py', recursive=True)
    if hits: return os.path.dirname(os.path.dirname(hits[0]))
    return os.getcwd() if os.path.exists('aegis/__init__.py') else None
root = find_aegis_root()
assert root, 'Attach the Aegis repo zip via Add Data, then re-run.'
sys.path.insert(0, root)
for m in [m for m in sys.modules if m=='aegis' or m.startswith('aegis.') or m=='eval' or m.startswith('eval.')]:
    del sys.modules[m]                       # drop any cached old modules so the new code is used
print('aegis repo at:', root)
!pip -q install datasets transformers torch sentence-transformers wandb 2>/dev/null
print('setup done')

## Secrets (optional, unlock more)
Add as **Kaggle Secrets** (Add-ons -> Secrets) - read automatically:

- **`HF_TOKEN`** -> loads the **gated** datasets. Click **Agree** once on each page: [AdvBench](https://huggingface.co/datasets/walledai/AdvBench) · [HarmBench](https://huggingface.co/datasets/walledai/HarmBench) · [WildGuardMix](https://huggingface.co/datasets/allenai/wildguardmix)
- **`WANDB_API_KEY`** -> logs to **Weights & Biases** (project `aegis-llm-defense`).

## Baselines + robustness (logs to W&B if WANDB_API_KEY is set)

In [ ]:
from eval.run_baselines import run
rows, robust = run(wandb_log=True)   # add use_guard=True for an open guard (GPU)

## Next
Numbers P2/P3 must beat. See `docs/Aegis_Design_and_Roadmap`.